# 02 · Exact Method: Cardinality-Constrained Portfolio Optimisation as a MILP

**Question:** which ≤ 5 of 457 S&P 500 stocks should we buy with $10,000, and how much of each, to reach a **20% expected annual return with the minimum risk**?

**Method:** a **Mixed-Integer Linear Program** (cardinality-constrained Mean-Absolute-Deviation model), solved to proven optimality, or to a certified gap, by **LP-based Branch & Bound** (HiGHS solver through PuLP; CBC as a cross-check).

**Contents**
1. Setup and problem parameters
2. Loading the problem instance (cleaned data from notebook 01)
3. Mathematical formulation
4. Implementation: building the MILP in PuLP
5. Implementation: solving it and reading the solver's proof (bound and gap)
6. A method-independent evaluator (shared with the heuristic)
7. Worked example: LP relaxation vs Branch & Bound on 10 stocks
8. Verification: is the "exact" answer really optimal?
9. Experiments: full problem, scalability, solver comparison, efficient frontier, sensitivity to K
10. Out-of-sample validation (2017–2018)
11. Export for the comparison with the heuristic
12. Discussion and future work

> **Running time.** The full experiments take about 2.5 hours (some solves run for 30 minutes). Their results are saved in `results/exact/`. With `RUN_EXPERIMENTS = False` (default), the notebook **loads** those saved results and runs in about a minute. Set it to `True` to recompute everything from scratch.

## 1. Setup and problem parameters

In [1]:
import itertools
import json
import math
import time
from dataclasses import dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
import pulp
from scipy.optimize import linprog
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

pd.set_option("display.float_format", "{:.5f}".format)
print("PuLP", pulp.__version__, "| solvers available:", pulp.listSolvers(onlyAvailable=True))

PuLP 3.3.2 | solvers available: ['GLPK_CMD', 'PULP_CBC_CMD', 'HiGHS']


**Investor parameters (Option C).** Every number the model uses is defined once here.

| Symbol | Value | Meaning |
|---|---|---|
| $B$ | \$10,000 | budget |
| $K$ | 5 | maximum number of stocks held (the investor can only follow 5 companies) |
| $l$ | 5% | minimum position size if a stock is bought (no meaningless 0.1% holdings) |
| $u$ | 40% | maximum position size (forces diversification: at least 3 stocks) |
| $\rho$ | 20% / 52 | required expected **weekly** return (20% per year) |
| cash | ≤ 1% | whole-share plans may leave at most $100 unspent |

In [2]:
BUDGET = 10_000.0
MAX_ASSETS = 5                 # K
MIN_WEIGHT = 0.05              # l
MAX_WEIGHT = 0.40              # u
TARGET_ANNUAL_RETURN = 0.20    # rho * 52
MAX_CASH = 0.01                # whole-share variant only
PERIODS_PER_YEAR = 52          # weekly data

RUN_EXPERIMENTS = False        # True = recompute all experiments (~2.5 h); False = load saved results
TIME_LIMIT = 300               # seconds per solve in the experiments
MAIN_TIME_LIMIT = 1800         # seconds for the headline full-size solve

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results" / "exact"
FIGURES_DIR = ROOT / "results" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Same figure style as notebook 01 (colour-blind-safe palette, light grid)
BLUE, ORANGE, GREEN, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#52514e", "#e4e3df"
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRID,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID,
    "axes.titlelocation": "left", "axes.titleweight": "semibold", "axes.titlesize": 12,
    "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.prop_cycle": mpl.cycler(color=[BLUE, ORANGE, GREEN]),
    "legend.frameon": False, "lines.linewidth": 2,
})

def save(fig, name):
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / f"{name}.png")

## 2. Loading the problem instance
Notebook 01 cleaned the Kaggle data and saved it to `data/processed/`. Here we read:
* `weekly_returns_train.csv`: the matrix $r_{ti}$ (203 weeks × 457 stocks) that defines the model;
* `daily_close_clean.csv`: daily prices, used for the decision-date share prices $p_i$ and for the out-of-sample test.

In [4]:
train = pd.read_csv(PROCESSED_DIR / "weekly_returns_train.csv", index_col=0, parse_dates=True)
daily_prices = pd.read_csv(PROCESSED_DIR / "daily_close_clean.csv", index_col=0, parse_dates=True)
decision_prices = daily_prices.loc[:"2016-12-31"].iloc[-1]     # close on 2016-12-30

print(f"returns: {train.shape[0]} weeks x {train.shape[1]} stocks, "
      f"{train.index.min().date()} → {train.index.max().date()}")
print(f"decision date: {daily_prices.loc[:'2016-12-31'].index[-1].date()}")

returns: 203 weeks x 457 stocks, 2013-02-15 → 2016-12-30
decision date: 2016-12-30


**The `Instance` class** bundles everything that defines one problem: the return data, the prices, and the parameters $K, l, u, \rho$. Bundling them lets us create smaller or modified problems easily (random subsets for the scalability study, other values of $K$, other targets) with `replace(...)`.

It also computes the derived model parameters:
* `mu`: $\mu_i = \frac{1}{T}\sum_t r_{ti}$, the expected weekly return of each stock;
* `deviations`: $a_{ti} = r_{ti} - \mu_i$, how far each stock is from its mean in each week, which is the building block of MAD;
* `max_achievable_return`: the highest return any feasible portfolio can reach. Fill the best stocks up to $u$ each; greedy is optimal for this small LP. A target above this value is infeasible.

In [5]:
@dataclass
class Instance:
    returns: pd.DataFrame                  # T x N weekly returns r_ti
    prices: pd.Series                      # price p_i on the decision date
    K: int = MAX_ASSETS
    w_min: float = MIN_WEIGHT
    w_max: float = MAX_WEIGHT
    target: float = TARGET_ANNUAL_RETURN / PERIODS_PER_YEAR   # rho (weekly)
    budget: float = BUDGET
    max_cash: float = MAX_CASH

    def __post_init__(self):
        self.prices = self.prices.loc[self.returns.columns]    # keep prices aligned with stocks

    @property
    def tickers(self): return list(self.returns.columns)
    @property
    def N(self): return self.returns.shape[1]
    @property
    def T(self): return self.returns.shape[0]
    @property
    def mu(self): return self.returns.values.mean(axis=0)
    @property
    def deviations(self): return self.returns.values - self.mu

    def max_achievable_return(self):
        remaining, total = 1.0, 0.0
        for m in np.sort(self.mu)[::-1][: self.K]:
            w = min(self.w_max, remaining)
            total, remaining = total + w * m, remaining - w
            if remaining <= 1e-12:
                return total
        return -np.inf

    def subset(self, tickers, **changes):
        return replace(self, returns=self.returns[tickers], **changes)


def random_subinstance(inst, n, seed):
    """n randomly chosen stocks (reproducible via seed). A random subset may not contain
    stocks good enough for the 20% target, so the target is capped at 90% of what the
    subset can achieve - this keeps every sampled instance feasible."""
    rng = np.random.default_rng(seed)
    tickers = sorted(rng.choice(inst.tickers, size=n, replace=False))
    target = min(inst.target, 0.9 * inst.subset(tickers).max_achievable_return())
    return inst.subset(tickers, target=target)


inst = Instance(returns=train, prices=decision_prices)
print(f"N = {inst.N} stocks, T = {inst.T} weeks, K = {inst.K}, weights in [{inst.w_min:.0%}, {inst.w_max:.0%}], "
      f"target = {inst.target * 52:.0%}/yr (max achievable {inst.max_achievable_return() * 52:.0%}/yr)")

N = 457 stocks, T = 203 weeks, K = 5, weights in [5%, 40%], target = 20%/yr (max achievable 57%/yr)


## 3. Mathematical formulation

**Sets and indices**
* $i \in \{1,\dots,N\}$: candidate stocks ($N = 457$)
* $t \in \{1,\dots,T\}$: weekly scenarios in the estimation window ($T = 203$)

**Parameters (from the data)**
* $r_{ti}$: return of stock $i$ in week $t$
* $\mu_i = \frac{1}{T}\sum_t r_{ti}$: expected weekly return of stock $i$
* $a_{ti} = r_{ti} - \mu_i$: deviation of stock $i$ from its mean in week $t$
* $\rho$: required expected return ($20\%/52$ per week); $K = 5$; $l = 0.05$; $u = 0.40$

**Decision variables**
* $w_i \in [0, u]$: fraction of the budget invested in stock $i$ (continuous)
* $z_i \in \{0, 1\}$: 1 if stock $i$ is bought (binary)
* $d_t \ge 0$: absolute deviation of the portfolio return in week $t$ (auxiliary)

**Model (CC-MAD)**
$$
\begin{aligned}
\min_{w,z,d}\quad & \frac{1}{T}\sum_{t=1}^{T} d_t && \text{(mean absolute deviation = risk)}\\
\text{s.t.}\quad & d_t \ \ge\ \ \ \sum_{i} a_{ti} w_i && \forall t \quad (1)\\
& d_t \ \ge\ -\sum_{i} a_{ti} w_i && \forall t \quad (2)\\
& \sum_{i} w_i = 1 && (3)\ \text{fully invested}\\
& \sum_{i} \mu_i w_i \ \ge\ \rho && (4)\ \text{return target}\\
& \sum_{i} z_i \ \le\ K && (5)\ \text{at most 5 stocks}\\
& l\, z_i \ \le\ w_i \ \le\ u\, z_i && \forall i \quad (6)\ \text{linking}\\
& z_i \in \{0,1\},\ w_i \ge 0,\ d_t \ge 0
\end{aligned}
$$

**How each part works**
* **Objective.** The risk is the average size of the portfolio's weekly deviation from its own mean: $\frac1T\sum_t |\sum_i a_{ti}w_i|$.
* **(1)–(2) linearise the absolute value.** $d_t$ must be at least $+x$ and at least $-x$, so $d_t \ge |x|$. Because we *minimise* $d_t$, at the optimum $d_t = |x|$ exactly. This trick keeps the model linear, so PuLP can solve it.
* **(3)** all money is invested. **(4)** the expected return meets the target.
* **(5)** is the **cardinality constraint**. Only (5), together with the binaries, makes the problem combinatorial; without it the model is a plain LP.
* **(6) is a Big-M linking constraint** (here $M = u$). If $z_i = 0$ then $w_i = 0$: a stock cannot receive money unless it is "picked". If $z_i = 1$ then $l \le w_i \le u$.

**Whole-share variant (realistic purchase plan).** Let $p_i$ be the price on the decision date, $B$ the budget and $x_i \in \mathbb{Z}_{\ge 0}$ the number of shares bought. Add $B\,w_i = p_i\,x_i$, and replace (3) by $1 - c \le \sum_i w_i \le 1$, where $c$ = 1% is the cash left over from rounding to whole shares.

**Why MAD and not variance (Markowitz)?** Variance makes the objective *quadratic* (an MIQP), and PuLP handles **linear** models only. MAD (Konno & Yamazaki, 1991) keeps the risk–return trade-off with linear constraints, and it is more robust to the fat-tailed returns found in notebook 01.

**Problem class.** With constraint (5) the problem is **NP-hard** (Bienstock, 1996; Moral-Escudero et al., 2006). It contains subset selection: choosing ≤ 5 of 457 stocks gives about $1.6 \times 10^{11}$ subsets, and each needs its weights optimised.

## 4. Implementation: building the MILP in PuLP
`build_model` turns the formulation into a PuLP model **line by line**; the comments refer to the equation numbers above.

Two options are used later:
* `integer_shares=True` adds the whole-share variables $x_i$;
* `relax=True` makes $z_i$ (and $x_i$) continuous. This gives the **LP relaxation**, which is exactly what Branch & Bound solves at its first (root) node.

In [6]:
def build_model(inst, integer_shares=False, relax=False):
    N, T = inst.N, inst.T
    a, mu = inst.deviations, inst.mu
    binary = pulp.LpContinuous if relax else pulp.LpBinary
    integer = pulp.LpContinuous if relax else pulp.LpInteger

    m = pulp.LpProblem("CC_MAD_portfolio", pulp.LpMinimize)

    # decision variables
    w = [m.add_variable(f"w_{i}", lowBound=0, upBound=inst.w_max) for i in range(N)]       # weights
    z = [m.add_variable(f"z_{i}", lowBound=0, upBound=1, cat=binary) for i in range(N)]    # pick stock i?
    d = [m.add_variable(f"d_{t}", lowBound=0) for t in range(T)]                           # |deviation| in week t

    # objective: mean absolute deviation
    m += pulp.lpSum(d) * (1.0 / T), "MAD"

    # (1)-(2): d_t >= |sum_i a_ti w_i|
    for t in range(T):
        port_dev = pulp.LpAffineExpression([(w[i], a[t, i]) for i in range(N)])
        m += d[t] - port_dev >= 0, f"abs_pos_{t}"
        m += d[t] + port_dev >= 0, f"abs_neg_{t}"

    # (4) return target and (5) cardinality
    m += pulp.LpAffineExpression([(w[i], mu[i]) for i in range(N)]) >= inst.target, "return_target"
    m += pulp.lpSum(z) <= inst.K, "cardinality"

    # (6) linking: w_i = 0 unless z_i = 1, and then l <= w_i <= u
    for i in range(N):
        m += w[i] - inst.w_max * z[i] <= 0, f"link_upper_{i}"
        m += w[i] - inst.w_min * z[i] >= 0, f"link_lower_{i}"

    x = None
    if integer_shares:
        # whole shares: B * w_i = p_i * x_i, and at most max_cash left uninvested
        p = inst.prices.values
        x = [m.add_variable(f"x_{i}", lowBound=0, upBound=float(np.floor(inst.w_max * inst.budget / p[i])),
                            cat=integer) for i in range(N)]
        for i in range(N):
            m += inst.budget * w[i] - p[i] * x[i] == 0, f"shares_{i}"
        m += pulp.lpSum(w) <= 1, "budget"
        m += pulp.lpSum(w) >= 1 - inst.max_cash, "max_cash"
    else:
        m += pulp.lpSum(w) == 1, "budget"     # (3) fully invested

    return m, {"w": w, "z": z, "d": d, "x": x}


model, _ = build_model(inst)
pd.Series({
    "binary variables z": inst.N,
    "continuous variables w, d": inst.N + inst.T,
    "constraints": model.numConstraints(),
    "non-zeros in (1)-(2)": 2 * inst.N * inst.T,
    "candidate subsets |S| <= 5": f"{sum(math.comb(inst.N, k) for k in range(1, 6)):.3e}",
}).to_frame("full problem")

,full problem
binary variables z,457
"continuous variables w, d",660
constraints,1323
non-zeros in (1)-(2),185542
candidate subsets |S| <= 5,1.643e+11


## 5. Implementation: solving the model
**How the solver works: LP-based Branch & Bound** (Land & Doig, 1960), plus cutting planes and primal heuristics:
1. Solve the **LP relaxation** ($0 \le z_i \le 1$). Its value is a **lower bound** on the best possible risk.
2. If some $z_i$ is fractional, **branch**: create one sub-problem with $z_i = 0$ and one with $z_i = 1$.
3. **Prune** a node if its LP is infeasible, or its bound is no better than the best integer portfolio found so far (the *incumbent*), or its LP solution is already integral.
4. When every node is pruned, the incumbent is **proven optimal**.

At any moment the solver knows two numbers: the **incumbent** (the best portfolio found) and the **best bound** (no portfolio can be better than this). Their relative difference is the **optimality gap**:
$$\text{gap} = \frac{\text{incumbent} - \text{bound}}{\text{incumbent}}$$
If a time limit stops the search early, we still get a portfolio *and* a guarantee that it is at most *gap* % worse than the true optimum. A heuristic cannot give this guarantee.

**A PuLP pitfall we found.** PuLP's `status` says *"Optimal"* even when the time limit stopped the solver with only a feasible solution. The reliable field is `sol_status` (1 = proven optimal, 2 = feasible but not proven). For HiGHS we read the bound, gap and node count directly from the solver object; for CBC we parse its log.

In [7]:
SOL_STATUS = {
    pulp.LpSolutionOptimal: "optimal",
    pulp.LpSolutionIntegerFeasible: "feasible (time limit)",
    pulp.LpSolutionNoSolutionFound: "no solution found",
    pulp.LpSolutionInfeasible: "infeasible",
    pulp.LpSolutionUnbounded: "unbounded",
}
LOG_DIR = RESULTS_DIR / "logs"


def make_solver(name, time_limit, gap_rel=1e-6, msg=False):
    if name == "highs":
        return pulp.HiGHS(msg=msg, timeLimit=time_limit, gapRel=gap_rel)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    return pulp.PULP_CBC_CMD(msg=msg, timeLimit=time_limit, gapRel=gap_rel, logPath=str(LOG_DIR / "cbc_last.log"))


def solve(inst, solver="highs", time_limit=300, integer_shares=False, relax=False, msg=False):
    """Build + solve the model. Returns a dict with the portfolio and the solver's proof."""
    t0 = time.perf_counter()
    m, var = build_model(inst, integer_shares=integer_shares, relax=relax)
    build_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    m.solve(make_solver(solver, time_limit, msg=msg))
    solve_time = time.perf_counter() - t0

    if relax:   # an LP: either solved to optimality or not
        status = "optimal" if m.status == pulp.LpStatusOptimal else pulp.LpStatus[m.status]
    else:
        status = SOL_STATUS.get(m.sol_status, pulp.LpStatus[m.status])
    has_solution = status in ("optimal", "feasible (time limit)")

    w = np.array([max(v.value() or 0.0, 0.0) for v in var["w"]]) if has_solution else np.zeros(inst.N)
    w[w < 1e-9] = 0.0
    weights = pd.Series(w, index=inst.tickers)
    objective = pulp.value(m.objective) if has_solution else None
    shares = (pd.Series([round(v.value()) for v in var["x"]], index=inst.tickers)
              if integer_shares and has_solution else None)

    # proof: best bound, gap, explored nodes
    bound, nodes = None, None
    if relax:
        bound = objective
    elif solver == "highs":
        info = m.solverModel.getInfo()
        bound, nodes = info.mip_dual_bound, int(info.mip_node_count)
    else:  # CBC: read its log
        import re
        log = (LOG_DIR / "cbc_last.log").read_text()
        b, n = re.search(r"Lower bound:\s+([-\d.eE+]+)", log), re.search(r"Enumerated nodes:\s+(\d+)", log)
        bound, nodes = (float(b.group(1)) if b else objective), (int(n.group(1)) if n else None)
    gap = max(objective - bound, 0.0) / objective if objective and bound is not None else None

    return {
        "status": status, "proven_optimal": status == "optimal", "mad": objective,
        "best_bound": bound, "gap": gap, "solve_time": solve_time, "build_time": build_time,
        "nodes": nodes, "weights": weights, "portfolio": weights[weights > 0].sort_values(ascending=False),
        "shares": shares, "solver": solver,
        "n_variables": len(m.variables()), "n_integer": sum(v.cat == pulp.LpInteger for v in m.variables()),
        "n_constraints": m.numConstraints(),
    }

## 6. A method-independent evaluator
`evaluate_weights` scores **any** portfolio (from this MILP or from the heuristic) with the same objective and the same feasibility rules. This keeps the comparison between methods fair. It is deliberately written **without** PuLP, so it is an independent check of the solver's answer.

> **For the heuristic notebook:** copy this function (and the parameter cell) unchanged, so both methods are scored identically.

In [8]:
def evaluate_weights(weights, inst, tol=1e-6):
    """weights: array aligned with inst.tickers, or {ticker: weight} dict."""
    if isinstance(weights, dict):
        weights = np.array([weights.get(t, 0.0) for t in inst.tickers])
    w = np.asarray(weights, dtype=float)
    held = w > tol
    exp_ret = float(inst.mu @ w)

    violations = []
    if held.sum() > inst.K:
        violations.append(f"holds {held.sum()} > {inst.K} stocks")
    if (w[held] < inst.w_min - tol).any():
        violations.append("weight below minimum")
    if (w > inst.w_max + tol).any():
        violations.append("weight above maximum")
    if w.sum() > 1 + tol or (w < -tol).any():
        violations.append("budget exceeded")
    if w.sum() < 1 - inst.max_cash - tol:
        violations.append(f"more than {inst.max_cash:.0%} left as cash")
    if exp_ret < inst.target - tol:
        violations.append("return target not met")

    return {
        "mad": float(np.abs(inst.deviations @ w).mean()),
        "expected_annual_return": exp_ret * PERIODS_PER_YEAR,
        "annual_volatility": float(np.std(inst.returns.values @ w, ddof=1) * np.sqrt(PERIODS_PER_YEAR)),
        "n_assets": int(held.sum()),
        "invested": float(w.sum()),
        "feasible": not violations,
        "violations": violations,
    }

## 7. Worked example: LP relaxation vs Branch & Bound on 10 stocks
We take 10 random stocks and solve:
* the **LP relaxation** (what B&B sees at the root node);
* the full **MILP**.

In [9]:
toy = random_subinstance(inst, n=10, seed=0)

lp_model, lp_vars = build_model(toy, relax=True)
lp_model.solve(make_solver("highs", 60))
lp = pd.DataFrame({"w": [v.value() for v in lp_vars["w"]], "z": [v.value() for v in lp_vars["z"]]},
                  index=toy.tickers)
mip = solve(toy)

table = pd.DataFrame({"LP relaxation: w": lp["w"], "LP relaxation: z": lp["z"],
                      "MILP: w": mip["weights"], "MILP: z": (mip["weights"] > 0).astype(int)}).round(3)
display(table)
lp_mad = pulp.value(lp_model.objective)
print(f"LP relaxation MAD (lower bound): {lp_mad:.5f} - stocks with w>0: {(lp['w'] > 1e-9).sum()}")
print(f"MILP optimum MAD               : {mip['mad']:.5f} - stocks held: {len(mip['portfolio'])} "
      f"(B&B nodes: {mip['nodes']}, {mip['solve_time']:.2f}s)")
print(f"gap closed by branching        : {(mip['mad'] - lp_mad) / mip['mad']:.1%}")

,LP relaxation: w,LP relaxation: z,MILP: w,MILP: z
ACN,0.12600,0.31500,0.16000,1
AFL,0.04800,0.12100,0.07600,1
AMP,0.06900,0.17300,0.00000,0
CCI,0.03700,0.09200,0.00000,0
DG,0.18300,0.45800,0.21300,1
EFX,0.40000,1.00000,0.40000,1
JEC,0.00000,0.00000,0.00000,0
MOS,0.00000,0.00000,0.00000,0
SJM,0.11700,0.29300,0.15100,1
SRE,0.01900,0.04800,0.00000,0


LP relaxation MAD (lower bound): 0.01407 - stocks with w>0: 8
MILP optimum MAD               : 0.01424 - stocks held: 5 (B&B nodes: 5, 0.41s)
gap closed by branching        : 1.2%


**What we see.**
* The LP relaxation invests in **8** stocks, more than the 5 allowed, with fractional $z_i$.
* Branch & Bound forces each $z_i$ to 0 or 1 and finds the best 5-stock portfolio. Its risk is higher, because the restriction costs something.

**Why the relaxation is weak (key insight for large problems).** Look at the column $z$: every relaxed value is exactly $z_i = w_i / u$ (e.g. 0.126 / 0.4 = 0.315). That is the cheapest way to satisfy (6). Then
$$\sum_i z_i = \frac{1}{u}\sum_i w_i = \frac{1}{0.4} = 2.5 \ \le\ K = 5,$$
so **the cardinality constraint (5) never binds at the root node**. The root bound is just the risk of the best *unrestricted* portfolio, and the solver gets no help from (5) until it branches. Section 9 shows that this makes large instances hard to *prove* optimal.